In [13]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train = df_train.dropna(axis=0) # e doar un rand cu nulluri nu are sens sa fac imputing
df_test = df_test.dropna(axis=0)

df_train.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6t6PZ4020pWJJlpZMtkx2l,Tulipa Ruiz,Habilidades Extraordinárias,Estardalhaço,39,172834,False,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,mpb
1,7oY7CNk9rtJShki9P25ohi,Yanni,In My Time,Before I Go,41,273640,False,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,new-age
2,3aVj8MbLsAwi2ZIkq5b5OY,Paffendorf,Lalala Girl,Lalala Girl - Short Version,28,212403,False,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,techno
3,6VvYKjGL96kwsJJbzbSeYi,Floyd Cramer,Piano Masterpieces,The Old Piano Roll Blues - (Previously titled ...,17,135040,False,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,honky-tonk
4,3EEJ63VT1FN3sQod8vd4e8,Dirt Monkey,(RE)EVOLUTION VOL. 2,(RE)EVOLUTION VOL. 2,24,1556883,False,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,dub


In [14]:
#preprocessing

df_train.explicit = df_train.explicit.map({False: 0, True: 1})
df_test.explicit = df_test.explicit.map({False: 0, True: 1})

to_encode = ['artists', 'track_genre', 'album_name']

from sklearn.preprocessing import LabelEncoder

for col in to_encode:
    enc = LabelEncoder()
    all_labels = pd.concat([df_train[col], df_test[col]])
    enc.fit(all_labels)
    df_train[col] = enc.transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

df_train.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6t6PZ4020pWJJlpZMtkx2l,28545,15784,Estardalhaço,39,172834,0,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,74
1,7oY7CNk9rtJShki9P25ohi,30067,18273,Before I Go,41,273640,0,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,75
2,3aVj8MbLsAwi2ZIkq5b5OY,20882,20900,Lalala Girl - Short Version,28,212403,0,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,109
3,6VvYKjGL96kwsJJbzbSeYi,9624,28863,The Old Piano Roll Blues - (Previously titled ...,17,135040,0,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,52
4,3EEJ63VT1FN3sQod8vd4e8,7558,92,(RE)EVOLUTION VOL. 2,24,1556883,0,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,28


In [15]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[\d+]', '', text)
    return text

df_train.track_name = df_train.track_name.apply(preprocess)
df_test.track_name = df_test.track_name.apply(preprocess)

df_train.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6t6PZ4020pWJJlpZMtkx2l,28545,15784,estardalhaço,39,172834,0,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,74
1,7oY7CNk9rtJShki9P25ohi,30067,18273,before i go,41,273640,0,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,75
2,3aVj8MbLsAwi2ZIkq5b5OY,20882,20900,lalala girl short version,28,212403,0,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,109
3,6VvYKjGL96kwsJJbzbSeYi,9624,28863,the old piano roll blues previously titled pi...,17,135040,0,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,52
4,3EEJ63VT1FN3sQod8vd4e8,7558,92,reevolution vol,24,1556883,0,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,28


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(analyzer='char', ngram_range=((3, 5)))
all_titles = pd.concat([df_train.track_name, df_test.track_name])
vect.fit(all_titles)
X_titles = vect.transform(df_train.track_name)
X_titles_final = vect.transform(df_test.track_name)

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=100, random_state=42)
X_titles = svd.fit_transform(X_titles)
X_titles_final = svd.transform(X_titles_final)

X_titles

array([[ 0.0066762 ,  0.0076454 , -0.00189892, ..., -0.01083619,
         0.00502569,  0.00564804],
       [ 0.0267844 ,  0.0301865 , -0.01062847, ...,  0.00487524,
         0.0129007 ,  0.03630605],
       [ 0.03747403,  0.03972675, -0.01187579, ..., -0.03126413,
        -0.02993511,  0.01821632],
       ...,
       [ 0.02458401,  0.02487947, -0.01692973, ..., -0.01725069,
         0.01683279,  0.00068125],
       [ 0.63293198, -0.41551643, -0.04028287, ..., -0.01879416,
         0.0158849 , -0.03595529],
       [ 0.00737758,  0.0095462 , -0.00358875, ..., -0.01125129,
         0.00677253,  0.01679691]])

In [17]:
features = ['artists', "album_name", 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre']
target = 'popularity'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_features = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_features_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,artists,album_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,28545,15784,172834,0,0.570,0.817,8,-4.657,1,0.0701,0.21400,0.0000,0.0771,0.776,174.241,3,74
1,30067,18273,273640,0,0.303,0.135,3,-19.767,1,0.0308,0.94700,0.9440,0.0907,0.078,94.762,4,75
2,20882,20900,212403,0,0.611,0.884,0,-6.360,0,0.0367,0.00093,0.2670,0.0889,0.772,139.001,4,109
3,9624,28863,135040,0,0.713,0.653,0,-13.365,1,0.0349,0.95000,0.8840,0.1030,0.921,152.758,4,52
4,7558,92,1556883,0,0.528,0.780,2,-5.549,1,0.1210,0.02770,0.0168,0.1850,0.329,125.065,4,28


In [18]:
import numpy as np

X = np.hstack([X_features, X_titles])
X_final = np.hstack([X_features_final, X_titles_final])

X

array([[ 1.41087494e+00, -5.02664378e-01, -5.22905034e-01, ...,
        -1.08361912e-02,  5.02568599e-03,  5.64803600e-03],
       [ 1.57878064e+00, -3.17302505e-01,  4.35428621e-01, ...,
         4.87523766e-03,  1.29007021e-02,  3.63060454e-02],
       [ 5.65499526e-01, -1.21663438e-01, -1.46733929e-01, ...,
        -3.12641287e-02, -2.99351063e-02,  1.82163172e-02],
       ...,
       [ 6.14260577e-01,  7.15183230e-01,  3.67607734e-01, ...,
        -1.72506910e-02,  1.68327852e-02,  6.81246900e-04],
       [ 4.24070415e-01, -1.50290355e+00, -3.39178309e-01, ...,
        -1.87941624e-02,  1.58848971e-02, -3.59552930e-02],
       [ 6.95786406e-01,  1.20037110e+00,  3.37442936e-01, ...,
        -1.12512886e-02,  6.77252614e-03,  1.67969104e-02]])

In [19]:
from sklearn.linear_model import LinearRegression # kinda ass
from sklearn.ensemble import RandomForestRegressor # dureaza mult prea mult
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score

model = LGBMRegressor(random_state=42, verbose=-1)

cross_val_score(model, X, y, cv=2, scoring='neg_mean_absolute_error').mean()

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


-13.41043762268648

In [20]:
model.fit(X, y)
y_preds = model.predict(X_final)

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [21]:
submission = pd.DataFrame({
    "track_id": df_test.track_id,
    "popularity": y_preds
})

submission

,track_id,popularity
0,4zI0WV1Qz7c7BJfQWVFBNO,30.023587
1,7Meqm9jgQXhh08vdnbrvfY,26.121416
2,4JnX8tY9me7jHqDwgifhwU,49.950508
3,7me5d5XmtCCfbDk3SsyXqM,45.033748
4,30Hm5tUVPVoJzGIEBchOT3,11.303920
...,...,...
28495,1U3GdQLdncM09zLxAZr75a,28.280751
28496,6DkWvMJZjYQdeat4itVt97,33.016065
28497,622fSIVOm6SPcLPNOoYeJn,37.724175
28498,7JJyuNGhAiPO1UHZcMITd8,27.208843


In [22]:
submission.to_csv("submission.csv", index=False)